# MongoDB Aggregation Pipeline & Cluster Shard Routing

Interactive hands-on sandbox exploring internal storage mechanics, algorithmic data structures, and architectural invariants.


In [ ]:
import sys
from pathlib import Path

# Prepend project_solution to sys.path so Track A internal engines import cleanly
ps_dir = Path('.').resolve() / 'project_solution'
if str(ps_dir) not in sys.path:
    sys.path.insert(0, str(ps_dir))

print(f'Python runtime: {sys.version.split()[0]}')
print('Loaded internal mechanics for: Module_11_MongoDB_Aggregations_Replicas_Sharding')


## 1. Engine Initialization & Setup

Importing the module's Track A internal simulation engine and instantiating state.


In [ ]:
from mongo_aggregation_engine import AggregationPipeline, ShardRouter

# Initialize Sample E-Commerce Order Records
orders = [
    {"order_id": 1, "customer": "Alice", "status": "COMPLETED", "items": [{"prod": "Laptop", "price": 1200}, {"prod": "Mouse", "price": 25}]},
    {"order_id": 2, "customer": "Bob", "status": "CANCELLED", "items": [{"prod": "Monitor", "price": 300}]},
    {"order_id": 3, "customer": "Charlie", "status": "COMPLETED", "items": [{"prod": "Keyboard", "price": 100}, {"prod": "Mouse", "price": 25}]},
    {"order_id": 4, "customer": "Alice", "status": "COMPLETED", "items": [{"prod": "Laptop", "price": 1200}]},
]

print(f"Loaded {len(orders)} order documents for aggregation.")


## 2. Core Architectural Operations & State Mutation

Executing data mutations, transactions, or indexing procedures.


In [ ]:
# Build Aggregation Pipeline: $match -> $unwind -> $group -> $sort
pipeline = (
    AggregationPipeline(orders)
    .match({"status": "COMPLETED"})
    .unwind("items")
    .group("items.prod", {
        "total_revenue": ("$sum", "items.price"),
        "units_sold": ("$count", "items.prod")
    })
    .sort("total_revenue", descending=True)
)
results = pipeline.execute()

print("Aggregation Pipeline Output:")
for r in results:
    print(f"  Product: {r['_id']:<15} Units Sold: {r['units_sold']} Total Revenue: ${r['total_revenue']}")


## 3. Performance Micro-Benchmarking & Invariant Verification

Evaluating execution latency, cache hits, or computational trade-offs.


In [ ]:
# Shard Router: Hash-Based Partition Routing
router = ShardRouter(num_shards=4)
router.insert({"customer_id": "cust_101", "name": "Alice"})
router.insert({"customer_id": "cust_202", "name": "Bob"})

shard_alice = router.get_shard_id("cust_101")
shard_bob = router.get_shard_id("cust_202")
print(f"Targeted query routed to shards -> Alice: Shard {shard_alice}, Bob: Shard {shard_bob}")


## 4. Architectural Invariant Verification

Asserting mathematical correctness and durability invariants.


In [ ]:
# Verify Invariants
assert results[0]["_id"] == "Laptop"
assert results[0]["total_revenue"] == 2400 # 2 Laptops sold
assert results[0]["units_sold"] == 2
print("[+] MongoDB Aggregation Pipeline and Sharding invariants verified successfully!")


## Summary & Operational DBRE Best Practices

1. **Never bypass serialization contracts:** Always enforce binary-safe schemas and validated boundaries.
2. **Monitor buffer and memory allocations:** Understand the latency cliff when in-memory structures spill to disk.
3. **Ensure idempotency across replication tiers:** Distributed operations must survive retries without corrupting state.
